# SetWise Kinetic Profiling v6

MM-Fit windowed classifier baseline for the SetWise v1 classifier path. This notebook intentionally keeps the model simple: prepared 5-second MM-Fit windows go into a compact 1D CNN, and the validation target is weighted cross-entropy converted to bits (`val_bpb`). Rep counting is not trained here.

## 1. Setup

`QUICK_RUN=True` is the default so the notebook can be smoke-tested locally or through the VS Code Colab extension. Set `QUICK_RUN=False` for the real A100 run.

In [ ]:
import copy
import csv
import json
import io
import math
import random
import subprocess
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
QUICK_RUN = True
QUICK_TRAIN = 2048
QUICK_VAL = 1024
QUICK_TEST = 1024

EPOCHS = 1 if QUICK_RUN else 20
PATIENCE = 2 if QUICK_RUN else 5
BATCH_SIZE = 64 if QUICK_RUN else (256 if torch.cuda.is_available() else 64)
LR = 1e-3
WEIGHT_DECAY = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = torch.cuda.is_available()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, quick_run={QUICK_RUN}, batch_size={BATCH_SIZE}, epochs={EPOCHS}")
if QUICK_RUN:
    print("QUICK_RUN is only a smoke test. Set QUICK_RUN=False for meaningful validation/test metrics.")

## 2. Load Prepared Windows

The notebook checks local repo paths first, then the expected Google Drive prepared directory.

In [ ]:
ARTIFACT_STEM = "setwise_mmfit_windows_50hz_5s_stride0p2"
DRIVE_ROOT = Path("/content/drive/MyDrive/SetwiseKineticDatasets")
DRIVE_PREPARED_DIR = DRIVE_ROOT / "prepared"
DRIVE_MMFIT_DIR = DRIVE_ROOT / "mm-fit-dataset"
PREPARED_DIR_OVERRIDE = None  # Example: Path("/content/drive/MyDrive/SetwiseKineticDatasets/prepared")
AUTO_PREPARE_IF_MISSING = True


def maybe_mount_drive():
    drive_root = Path("/content/drive")
    if not Path("/content").exists() or drive_root.exists():
        return
    try:
        from google.colab import drive
    except Exception:
        return
    print("Mounting Google Drive at /content/drive ...")
    drive.mount("/content/drive")


def artifact_paths(prepared_dir):
    return (
        prepared_dir / f"{ARTIFACT_STEM}_X.npy",
        prepared_dir / f"{ARTIFACT_STEM}_labels.npz",
        prepared_dir / f"{ARTIFACT_STEM}_metadata.json",
    )


def artifacts_exist(prepared_dir):
    return all(path.exists() for path in artifact_paths(prepared_dir))


MMFIT_ACTIONS = [
    "squats",
    "lunges",
    "bicep_curls",
    "situps",
    "pushups",
    "tricep_extensions",
    "dumbbell_rows",
    "jumping_jacks",
    "dumbbell_shoulder_press",
    "lateral_shoulder_raises",
]
MMFIT_ACTION_TO_INDEX = {name: idx for idx, name in enumerate(MMFIT_ACTIONS)}
TRAIN_WORKOUTS = {"w01", "w02", "w03", "w04", "w06", "w07", "w08", "w16", "w17", "w18"}
VAL_WORKOUTS = {"w14", "w15", "w19"}
TEST_WORKOUTS = {"w00", "w05", "w12", "w13", "w20"}
FEATURE_NAMES = ["acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]


def split_name(workout_id):
    if workout_id in TRAIN_WORKOUTS:
        return "train"
    if workout_id in VAL_WORKOUTS:
        return "val"
    if workout_id in TEST_WORKOUTS:
        return "test"
    return "unused"


def quote_drive_query_value(value):
    return str(value).replace("'", "\\'")


def gsheet_file_id_from_shortcut_json(gsheet_path):
    try:
        with open(gsheet_path, "r") as handle:
            data = json.load(handle)
    except OSError:
        return None

    for key in ("doc_id", "resource_id", "id"):
        value = data.get(key)
        if value:
            text = str(value)
            return text.split(":")[-1]
    url = str(data.get("url", ""))
    marker = "/d/"
    if marker in url:
        return url.split(marker, 1)[1].split("/", 1)[0]
    return None


def find_gsheet_file_id_by_name(gsheet_path, service):
    sheet_name = Path(gsheet_path).stem
    escaped_name = quote_drive_query_value(sheet_name)
    query = (
        f"name = '{escaped_name}' and "
        "mimeType = 'application/vnd.google-apps.spreadsheet' and trashed = false"
    )
    response = service.files().list(
        q=query,
        spaces="drive",
        fields="files(id,name,parents,mimeType)",
        pageSize=10,
    ).execute()
    files = response.get("files", [])
    if not files:
        raise FileNotFoundError(
            f"Could not locate Google Sheet named {sheet_name!r} through the Drive API. "
            f"The mounted shortcut was {gsheet_path}, but Colab could not open it directly."
        )
    if len(files) > 1:
        print(f"Found {len(files)} Google Sheets named {sheet_name}; using the first Drive API match.")
    return files[0]["id"]


def export_gsheet_as_csv(gsheet_path):
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except Exception as exc:
        raise RuntimeError(
            f"{gsheet_path} is a Google Sheets shortcut, not a CSV file. "
            "Run this notebook in Colab with Drive access, or replace .gsheet label shortcuts with exported .csv files."
        ) from exc

    auth.authenticate_user()
    service = build("drive", "v3")
    file_id = gsheet_file_id_from_shortcut_json(gsheet_path)
    if file_id is None:
        file_id = find_gsheet_file_id_by_name(gsheet_path, service)
    request = service.files().export_media(fileId=file_id, mimeType="text/csv")
    payload = request.execute()
    return payload.decode("utf-8")


def load_mmfit_labels(label_path):
    if str(label_path).endswith(".gsheet"):
        text = export_gsheet_as_csv(label_path)
        reader = csv.reader(io.StringIO(text))
    else:
        handle = open(label_path, "r", newline="")
        reader = csv.reader(handle)

    rows = []
    try:
        for row in reader:
            if not row or not row[0].strip().lstrip("-").isdigit():
                continue
            rows.append((int(row[0]), int(row[1]), int(row[2]), row[3]))
    finally:
        if "handle" in locals():
            handle.close()
    return rows



def find_workout_dirs(mmfit_raw_dir):
    root = Path(mmfit_raw_dir)
    direct = sorted(path for path in root.iterdir() if path.is_dir() and path.name.startswith("w") and len(path.name) == 3)
    if direct:
        return direct
    discovered = sorted({path.parent for path in root.rglob("w*_labels.csv") if path.parent.name.startswith("w")})
    if discovered:
        return discovered
    return sorted(path for path in root.rglob("w[0-9][0-9]") if path.is_dir())


def find_workout_file(workout, workout_id, suffix):
    search_suffixes = [suffix]
    if suffix == "labels.csv":
        search_suffixes.append("labels.gsheet")

    root = workout.parent
    for candidate_suffix in search_suffixes:
        expected = workout / f"{workout_id}_{candidate_suffix}"
        if expected.exists():
            return expected
        matches = sorted(workout.rglob(f"{workout_id}_{candidate_suffix}"))
        if matches:
            return matches[0]
        matches = sorted(root.rglob(f"{workout_id}_{candidate_suffix}"))
        if matches:
            return matches[0]

    visible = sorted(child.name for child in workout.iterdir())[:30] if workout.exists() and workout.is_dir() else []
    raise FileNotFoundError(
        f"Missing {workout_id}_{suffix} under {workout}. "
        f"First visible entries there: {visible}"
    )


def unique_sorted_time_values(timestamps_ms, values):
    order = np.argsort(timestamps_ms)
    sorted_times = timestamps_ms[order].astype(np.float64)
    sorted_values = values[order].astype(np.float32, copy=False)
    unique_times, unique_indices = np.unique(sorted_times, return_index=True)
    return unique_times, sorted_values[unique_indices]


def resample_mmfit_frames(acc_data, gyr_data, target_hz):
    start_ms = float(min(acc_data[:, 1].min(), gyr_data[:, 1].min()))
    end_ms = float(max(acc_data[:, 1].max(), gyr_data[:, 1].max()))
    duration_s = max((end_ms - start_ms) / 1000.0, 1.0 / target_hz)
    target_length = max(1, int(math.floor(duration_s * target_hz)) + 1)
    target_times_ms = start_ms + (np.arange(target_length, dtype=np.float64) / target_hz) * 1000.0
    frame_times, frame_values = unique_sorted_time_values(acc_data[:, 1], acc_data[:, 0:1])
    frames = np.interp(target_times_ms, frame_times, frame_values[:, 0]).astype(np.float32)
    return frames


def label_mmfit_timesteps(frames, label_rows):
    labels = np.full(frames.shape[0], len(MMFIT_ACTIONS), dtype=np.int64)
    for start_frame, end_frame, _reps, action_name in label_rows:
        class_id = MMFIT_ACTION_TO_INDEX.get(action_name)
        if class_id is None:
            continue
        labels[(frames >= start_frame) & (frames <= end_frame)] = class_id
    return labels


def compute_class_weights_np(y_values, split_values, num_classes):
    train_y = y_values[split_values == "train"]
    counts = np.bincount(train_y, minlength=num_classes).astype(np.float64)
    present = counts > 0
    weights = np.zeros(num_classes, dtype=np.float32)
    if np.any(present):
        weights[present] = float(train_y.size) / (float(np.sum(present)) * counts[present])
    return weights


def counter_to_dict(counter):
    return {str(key): int(counter[key]) for key in sorted(counter, key=lambda value: str(value))}


def nested_counter_to_dict(counter):
    return {str(key): counter_to_dict(value) for key, value in sorted(counter.items())}


def bootstrap_sidecars_from_existing_x(prepared_dir):
    x_path, labels_path, metadata_path = artifact_paths(prepared_dir)
    if not x_path.exists() or (labels_path.exists() and metadata_path.exists()):
        return False

    mmfit_raw_dir = find_mmfit_raw_dir()
    if mmfit_raw_dir is None:
        return False

    print("X.npy exists but sidecars are missing; deriving labels/metadata from raw MM-Fit.")
    print("raw MM-Fit:", mmfit_raw_dir)
    x_memmap = np.load(x_path, mmap_mode="r")
    target_hz = 50.0
    window_seconds = 5.0
    stride_seconds = 0.2
    window_samples = 250
    stride_samples = 10
    num_classes = len(MMFIT_ACTIONS) + 1

    y = []
    split_values = []
    workout_ids = []
    window_start_s = []
    window_end_s = []
    window_start_frame = []
    window_end_frame = []
    majority_fraction = []
    is_transition_window = []
    workout_window_counts = {}
    workout_sample_counts = {}

    workout_dirs = find_workout_dirs(mmfit_raw_dir)
    if not workout_dirs:
        raise RuntimeError(f"No MM-Fit workout directories found in {mmfit_raw_dir}")
    print("Found workout dirs:", [path.name for path in workout_dirs])

    for workout in workout_dirs:
        workout_id = workout.name
        acc_path = find_workout_file(workout, workout_id, "sw_l_acc.npy")
        gyr_path = find_workout_file(workout, workout_id, "sw_l_gyr.npy")
        label_path = find_workout_file(workout, workout_id, "labels.csv")

        acc_data = np.load(acc_path)
        gyr_data = np.load(gyr_path)
        frames = resample_mmfit_frames(acc_data, gyr_data, target_hz)
        timestep_labels = label_mmfit_timesteps(frames, load_mmfit_labels(label_path))
        split_value = split_name(workout_id)
        workout_sample_counts[workout_id] = int(frames.shape[0])
        count_for_workout = 0

        for start_idx in range(0, timestep_labels.shape[0] - window_samples + 1, stride_samples):
            end_idx = start_idx + window_samples
            window_labels = timestep_labels[start_idx:end_idx]
            counts = np.bincount(window_labels, minlength=num_classes)
            label = int(np.argmax(counts))
            majority = int(counts[label]) / float(window_samples)

            y.append(label)
            split_values.append(split_value)
            workout_ids.append(workout_id)
            window_start_s.append(float(start_idx / target_hz))
            window_end_s.append(float(end_idx / target_hz))
            window_start_frame.append(int(round(float(frames[start_idx]))))
            window_end_frame.append(int(round(float(frames[end_idx - 1]))))
            majority_fraction.append(majority)
            is_transition_window.append(majority < 1.0)
            count_for_workout += 1

        workout_window_counts[workout_id] = count_for_workout

    y = np.asarray(y, dtype=np.int64)
    split_arr = np.asarray(split_values, dtype="<U8")
    workout_id_arr = np.asarray(workout_ids, dtype="<U8")
    if y.shape[0] != x_memmap.shape[0]:
        raise RuntimeError(
            f"Derived {y.shape[0]} labels but X has {x_memmap.shape[0]} windows. "
            "Regenerate the full artifact set with prepare.py so X and sidecars match."
        )

    class_weight_train = compute_class_weights_np(y, split_arr, num_classes)
    sample_weight = class_weight_train[y].astype(np.float32)

    train_mask = split_arr == "train"
    channel_sum = np.zeros(x_memmap.shape[-1], dtype=np.float64)
    channel_sumsq = np.zeros(x_memmap.shape[-1], dtype=np.float64)
    count = 0
    chunk_size = 5000
    for start in range(0, x_memmap.shape[0], chunk_size):
        end = min(start + chunk_size, x_memmap.shape[0])
        mask = train_mask[start:end]
        if not np.any(mask):
            continue
        values = np.asarray(x_memmap[start:end][mask]).reshape(-1, x_memmap.shape[-1])
        channel_sum += values.sum(axis=0, dtype=np.float64)
        channel_sumsq += np.square(values, dtype=np.float64).sum(axis=0)
        count += values.shape[0]
    normalized_mean = (channel_sum / max(count, 1)).astype(np.float32)
    normalized_var = (channel_sumsq / max(count, 1)) - np.square(channel_sum / max(count, 1))
    normalized_std = np.sqrt(np.maximum(normalized_var, 1e-12)).astype(np.float32)

    label_names = MMFIT_ACTIONS + ["non_activity"]
    np.savez_compressed(
        labels_path,
        y=y,
        split=split_arr,
        workout_id=workout_id_arr,
        window_start_s=np.asarray(window_start_s, dtype=np.float32),
        window_end_s=np.asarray(window_end_s, dtype=np.float32),
        window_start_frame=np.asarray(window_start_frame, dtype=np.int64),
        window_end_frame=np.asarray(window_end_frame, dtype=np.int64),
        majority_fraction=np.asarray(majority_fraction, dtype=np.float32),
        is_transition_window=np.asarray(is_transition_window, dtype=np.bool_),
        sample_weight=sample_weight,
        class_weight_train=class_weight_train,
        label_names=np.asarray(label_names),
        feature_names=np.asarray(FEATURE_NAMES),
        target_hz=np.asarray(target_hz, dtype=np.float32),
        window_samples=np.asarray(window_samples, dtype=np.int64),
        stride_samples=np.asarray(stride_samples, dtype=np.int64),
        window_seconds=np.asarray(window_seconds, dtype=np.float32),
        stride_seconds=np.asarray(stride_seconds, dtype=np.float32),
        scaler_mean=normalized_mean.astype(np.float32),
        scaler_std=normalized_std.astype(np.float32),
    )

    split_counts = Counter(split_arr.tolist())
    label_counts = Counter(int(label) for label in y.tolist())
    label_counts_by_split = defaultdict(Counter)
    for split_value, label_value in zip(split_arr.tolist(), y.tolist()):
        label_counts_by_split[split_value][int(label_value)] += 1

    metadata = {
        "dataset": "mmfit",
        "view": "windows",
        "created_by": "SetWise_Kinetic_Profiling_v6 sidecar bootstrap",
        "x_path": str(x_path),
        "labels_path": str(labels_path),
        "target_hz": target_hz,
        "window_seconds": window_seconds,
        "stride_seconds": stride_seconds,
        "window_samples": window_samples,
        "stride_samples": stride_samples,
        "tensor_shape": [int(x_memmap.shape[0]), int(x_memmap.shape[1]), int(x_memmap.shape[2])],
        "feature_names": FEATURE_NAMES,
        "label_names": label_names,
        "non_activity_label": "non_activity",
        "input_paths": {"mmfit_dir": str(mmfit_raw_dir), "whales_dir": None},
        "split_policy": "split_repo_unseen; w09/w10/w11 retained as unused",
        "split_counts": counter_to_dict(split_counts),
        "label_counts": counter_to_dict(label_counts),
        "label_counts_by_split": nested_counter_to_dict(label_counts_by_split),
        "workout_window_counts": {key: int(value) for key, value in sorted(workout_window_counts.items())},
        "workout_sample_counts_50hz": {key: int(value) for key, value in sorted(workout_sample_counts.items())},
        "normalization": {
            "note": "X.npy was already present; sidecar bootstrap measured train normalized channel mean/std from X.",
            "train_normalized_mean": [float(value) for value in normalized_mean],
            "train_normalized_std": [float(value) for value in normalized_std],
            "leakage_guard": "Sidecar bootstrap uses labels/splits only; it does not refit or rewrite X.npy.",
        },
        "class_weight_train": [float(value) for value in class_weight_train],
        "notes": [
            "Sidecars were reconstructed because X.npy existed without labels.npz/metadata.json.",
            "Each sample is a 5-second overlapping left-watch MM-Fit window.",
            "Window labels are assigned by majority timestep label.",
            "non_activity covers timesteps outside labeled exercise sets.",
            "Whales is omitted from this v1 classifier dataset.",
        ],
    }
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True))
    print("Wrote labels:", labels_path)
    print("Wrote metadata:", metadata_path)
    print("Split counts:", dict(split_counts))
    return True


def in_colab_path_space():
    return Path("/content").exists()


def find_prepare_script():
    candidates = [
        Path("prepare.py"),
        Path.cwd() / "prepare.py",
        Path.cwd().parent / "prepare.py",
    ]
    if in_colab_path_space():
        candidates.append(DRIVE_ROOT / "prepare.py")
    for path in candidates:
        if path.exists():
            return path.resolve()
    return None


def find_mmfit_raw_dir():
    candidates = []
    if in_colab_path_space():
        candidates.append(DRIVE_MMFIT_DIR)
    candidates.extend([
        Path("mm-fit/mm-fit-dataset"),
        Path("mm-fit-dataset"),
        Path.cwd() / "mm-fit" / "mm-fit-dataset",
        Path.cwd().parent / "mm-fit" / "mm-fit-dataset",
    ])
    for path in candidates:
        if path.exists():
            return path.resolve()
    return None


def debug_listing(paths):
    lines = []
    for path in paths:
        lines.append(f"{path}: exists={path.exists()}")
        if path.exists() and path.is_dir():
            names = sorted(child.name for child in path.iterdir())[:30]
            lines.append("  first entries: " + ", ".join(names))
    return chr(10).join(lines)


def maybe_prepare_missing_artifacts(prepared_dir):
    if artifacts_exist(prepared_dir) or not AUTO_PREPARE_IF_MISSING:
        return

    x_path, labels_path, metadata_path = artifact_paths(prepared_dir)
    missing = [path.name for path in (x_path, labels_path, metadata_path) if not path.exists()]
    print("Prepared artifact sidecars are missing:", missing)
    if bootstrap_sidecars_from_existing_x(prepared_dir):
        return

    prepare_script = find_prepare_script()
    mmfit_raw_dir = find_mmfit_raw_dir()
    if prepare_script is None or mmfit_raw_dir is None:
        return

    prepared_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(prepare_script),
        "--dataset", "mmfit",
        "--view", "windows",
        "--window-seconds", "5",
        "--stride-seconds", "0.2",
        "--include-non-activity",
        "--target-hz", "50",
        "--mmfit-dir", str(mmfit_raw_dir),
        "--output-dir", str(prepared_dir),
        "--no-mount-drive",
    ]
    print("Regenerating missing prepared artifacts with:")
    print(" ".join(cmd))
    completed = subprocess.run(cmd, check=False, text=True, capture_output=True)
    print(completed.stdout[-4000:])
    if completed.returncode != 0:
        print(completed.stderr[-4000:])
        raise RuntimeError(f"prepare.py failed with exit code {completed.returncode}")


def find_prepared_dir():
    maybe_mount_drive()
    candidates = []
    if PREPARED_DIR_OVERRIDE is not None:
        candidates.append(Path(PREPARED_DIR_OVERRIDE))
    if in_colab_path_space():
        candidates.append(DRIVE_PREPARED_DIR)
    candidates.extend([
        Path("prepared"),
        Path("../prepared"),
        Path.cwd() / "prepared",
        Path.cwd().parent / "prepared",
    ])

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for prepared_dir in unique_candidates:
        maybe_prepare_missing_artifacts(prepared_dir)
        x_path, labels_path, metadata_path = artifact_paths(prepared_dir)
        if x_path.exists() and labels_path.exists() and metadata_path.exists():
            return prepared_dir.resolve(), x_path, labels_path, metadata_path

    searched = chr(10).join(str(p) for p in unique_candidates)
    debug_paths = [
        Path("/content/drive"),
        Path("/content/drive/MyDrive"),
        DRIVE_ROOT,
        DRIVE_PREPARED_DIR,
        DRIVE_MMFIT_DIR,
        Path("prepared"),
        Path("prepare.py"),
    ]
    command = (
        "python prepare.py --dataset mmfit --view windows --window-seconds 5 "
        "--stride-seconds 0.2 --include-non-activity --no-mount-drive "
        f"--mmfit-dir {DRIVE_MMFIT_DIR} --output-dir {DRIVE_PREPARED_DIR}"
    )
    raise FileNotFoundError(
        "Could not find the complete prepared MM-Fit window artifact set. The notebook needs all three files: "
        f"{ARTIFACT_STEM}_X.npy, {ARTIFACT_STEM}_labels.npz, and {ARTIFACT_STEM}_metadata.json."
        + chr(10) + "Searched:" + chr(10) + searched
        + chr(10) + chr(10) + "Directory visibility:" + chr(10) + debug_listing(debug_paths)
        + chr(10) + chr(10) + "If only the X.npy file is present, copy the labels.npz and metadata.json sidecars to the same folder or run:"
        + chr(10) + command
    )


prepared_dir, x_path, labels_path, metadata_path = find_prepared_dir()
print("prepared_dir:", prepared_dir)
print("X path:", x_path)
print("labels path:", labels_path)
print("metadata path:", metadata_path)

X = np.load(x_path, mmap_mode="r")
labels = np.load(labels_path)
with open(metadata_path, "r") as f:
    metadata = json.load(f)

y = labels["y"].astype(np.int64)
split = labels["split"].astype(str)
label_names = labels["label_names"].astype(str).tolist()
class_weight_train = labels["class_weight_train"].astype(np.float32)

print("X shape:", X.shape)
print("labels:", label_names)
print("splits:", {name: int((split == name).sum()) for name in np.unique(split)})
assert X.shape[1:] == (250, 6), X.shape
assert len(label_names) == 11
assert label_names[-1] == "non_activity"

## 3. DataLoaders

In [ ]:
rng = np.random.default_rng(SEED)

def split_indices(name):
    idx = np.flatnonzero(split == name)
    return idx.astype(np.int64)

def cap_indices(idx, cap):
    if not QUICK_RUN or len(idx) <= cap:
        return idx
    return np.sort(rng.choice(idx, size=cap, replace=False)).astype(np.int64)

train_idx = cap_indices(split_indices("train"), QUICK_TRAIN)
val_idx = cap_indices(split_indices("val"), QUICK_VAL)
test_idx = cap_indices(split_indices("test"), QUICK_TEST)

print(f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

class WindowDataset(Dataset):
    def __init__(self, x_path, y, indices):
        self.X = np.load(x_path, mmap_mode="r")
        self.y = y
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        x = np.array(self.X[j], dtype=np.float32, copy=True)  # (250, 6)
        return torch.from_numpy(x), torch.tensor(int(self.y[j]), dtype=torch.long)

train_ds = WindowDataset(x_path, y, train_idx)
val_ds = WindowDataset(x_path, y, val_idx)
test_ds = WindowDataset(x_path, y, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=device.type == "cuda")
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=device.type == "cuda")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=device.type == "cuda")

xb, yb = next(iter(train_loader))
print("first batch X:", tuple(xb.shape), "y:", tuple(yb.shape))
assert xb.shape[1:] == (250, 6)

## 4. Model

A compact 1D CNN is enough for the first classifier iteration. It transposes `(B, 250, 6)` into PyTorch's `(B, 6, 250)` channel-first format inside `forward`.

In [ ]:
class SimpleWindowCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(6, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.transpose(1, 2)
        return self.classifier(self.features(x))

model = SimpleWindowCNN(num_classes=len(label_names)).to(device)
train_criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weight_train, dtype=torch.float32, device=device))
eval_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

with torch.no_grad():
    logits = model(xb.to(device))
print("forward output:", tuple(logits.shape))
assert logits.shape == (xb.shape[0], len(label_names))

## 5. Train

`bpb` is cross-entropy divided by `log(2)`, so lower is better. The checkpoint selected for testing is the epoch with the lowest validation `bpb`.

In [ ]:
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = train_criterion(logits, yb) if train else eval_criterion(logits, yb)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        batch_n = yb.numel()
        total_loss += float(loss.detach().cpu()) * batch_n
        total_correct += int((logits.argmax(dim=1) == yb).sum().detach().cpu())
        total_count += batch_n

    avg_loss = total_loss / max(total_count, 1)
    return {
        "loss": avg_loss,
        "bpb": avg_loss / math.log(2),
        "acc": total_correct / max(total_count, 1),
    }

best_state = None
best_val_bpb = float("inf")
best_epoch = -1
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)

    improved = val_metrics["bpb"] < best_val_bpb
    if improved:
        best_val_bpb = val_metrics["bpb"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"epoch {epoch:02d} | "
        f"train_bpb={train_metrics['bpb']:.4f} train_acc={train_metrics['acc']:.4f} | "
        f"val_bpb={val_metrics['bpb']:.4f} val_acc={val_metrics['acc']:.4f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"early stopping at epoch {epoch}; best_epoch={best_epoch}, best_val_bpb={best_val_bpb:.4f}")
        break

if best_state is not None:
    model.load_state_dict(best_state)
print(f"selected_epoch={best_epoch}, selected_val_bpb={best_val_bpb:.4f}")

## 6. Test Evaluation

In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    preds = []
    targets = []
    total_loss = 0.0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        logits = model(xb)
        loss = eval_criterion(logits, yb)
        batch_n = yb.numel()
        total_loss += float(loss.detach().cpu()) * batch_n
        total_count += batch_n
        preds.append(logits.argmax(dim=1).cpu().numpy())
        targets.append(yb.cpu().numpy())

    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    avg_loss = total_loss / max(total_count, 1)
    return targets, preds, {
        "loss": avg_loss,
        "bpb": avg_loss / math.log(2),
        "acc": float((preds == targets).mean()),
    }

y_true, y_pred, test_metrics = predict(test_loader)
print(f"test_bpb={test_metrics['bpb']:.4f} test_acc={test_metrics['acc']:.4f}")
def print_classification_report(y_true, y_pred, label_names):
    print(f"{'label':28s} {'precision':>9s} {'recall':>9s} {'f1':>9s} {'support':>8s}")
    print('-' * 68)
    for class_id, name in enumerate(label_names):
        tp = int(((y_true == class_id) & (y_pred == class_id)).sum())
        fp = int(((y_true != class_id) & (y_pred == class_id)).sum())
        fn = int(((y_true == class_id) & (y_pred != class_id)).sum())
        support = int((y_true == class_id).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        print(f"{name:28s} {precision:9.3f} {recall:9.3f} {f1:9.3f} {support:8d}")

print_classification_report(y_true, y_pred, label_names)

## Full Run Note

For the real Colab/A100 run, set `QUICK_RUN = False` in the setup cell and rerun the notebook. Keep the first comparison simple: if a more complex model does not materially reduce validation `bpb`, keep this CNN.